# Boltz-1 MHC-I fine-tune on a free cloud T4

TACC access did not come through, so this is the fallback: a **16 GB T4**, which is
what both Google Colab's free tier and Kaggle's free tier hand out. The laptop
4060 has 8 GB and OOMs inside triangular attention even with the trunk frozen
(`reports/GPU_REQUIREMENTS.md`). 16 GB is the difference between that and a run.

This notebook works unchanged on **Colab** and on **Kaggle**. It detects which one
it is on and adjusts paths, storage and the resume story.

**Kaggle is the better host for this**, and it is worth knowing why before you pick:

| | Colab free | Kaggle free |
|---|---|---|
| GPU | T4 16 GB, *when available* | P100 16 GB, or **T4 x2** (pick T4 x2) |
| Session cap | 12 h, idle-disconnects at ~90 min | 12 h |
| Weekly budget | undocumented, ~15-30 h, varies with usage history | **30 h, stated** |
| Runs with the tab closed | no | **yes** -- "Save & Run All (Commit)" |
| Persistence | your Google Drive | Kaggle Datasets / notebook output |

The last two rows are what matter for a job measured in hours. On Colab you must
babysit the tab; on Kaggle you commit and walk away.

**If you are on Kaggle, choose the `GPU T4 x2` accelerator, not `GPU P100`.**
The P100 is Pascal (sm_60), which recent PyTorch wheels no longer build kernels
for. This notebook uses one of the two T4s; the second is left alone, because
sharding this model across both is a real piece of work (DeepSpeed/FSDP) and not
where the next result comes from.

### Before you start

Run `python src/make_cloud_bundle.py` **on the laptop** once. It produces a
~575 MB `mhc1-data-bundle.tar.gz` holding `data/processed/` and `data/msa/` --
the output of milestone 1, and the only part of `data/` that a cloud session
cannot cheaply regenerate. Then:

* **Colab** -- upload it to Drive, e.g. `MyDrive/mhc1/mhc1-data-bundle.tar.gz`.
* **Kaggle** -- create a Dataset from it and attach that dataset to this notebook.

The 3.8 GB of `data/assets/` is *not* in the bundle. This notebook re-downloads
the checkpoint and the symmetry pickle from their original hosts, which is faster
than pushing them through Drive.

## 1. What did we actually get?

In [ ]:
# Identify the host and the card before doing anything expensive.
import os, subprocess, sys, textwrap, shutil

ON_KAGGLE = os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

PLATFORM = "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local"
print(f"platform: {PLATFORM}")

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Colab: Runtime > Change runtime type > T4 GPU. "
        "Kaggle: Settings > Accelerator > GPU T4 x2."
    )

name = torch.cuda.get_device_name(0)
cap  = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"gpu:      {name}  (sm_{cap[0]}{cap[1]}, {vram:.1f} GiB)")
print(f"torch:    {torch.__version__}")

# The two failure modes worth catching now rather than 20 minutes in.
if cap[0] < 7:
    print(textwrap.dedent("""
        !! This is a Pascal card (P100). Recent torch wheels ship no sm_60 kernels,
        !! so the run will die with 'no kernel image is available'. On Kaggle,
        !! switch Accelerator to 'GPU T4 x2'.
    """))
if vram < 14:
    print(f"!! Only {vram:.1f} GiB. The budget in reports/CLOUD_GPU.md assumes ~16.")

print(f"cpus:     {os.cpu_count()}")
print(f"disk:     {shutil.disk_usage('/').free / 1024**3:.0f} GiB free")

# Internet is OFF by default on Kaggle and this notebook cannot work without it:
# it clones from GitHub and pulls 3.8 GB of assets. Catch it here, not in cell 4.
import socket
try:
    socket.create_connection(("pypi.org", 443), timeout=8).close()
    print("internet:  ok")
except OSError:
    print(
        "\n!! NO INTERNET. On Kaggle this is off by default.\n"
        "!!   right sidebar > Notebook options > Internet > On\n"
        "!! It requires a phone-verified account (Settings > Phone Verification).\n"
        "!! If the toggle is missing right after verifying, create a fresh\n"
        "!! notebook -- the option appears on new ones first."
    )

## 2. Persistent storage

Everything under the session's own filesystem dies when the session does. The run
directory -- checkpoints, logs, `val_state.json` -- has to live somewhere that
survives, or a 12 h cap means starting over every time.

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/mhc1-runs/t4")
    WORK    = Path("/content")
elif ON_KAGGLE:
    # /kaggle/working persists into the committed version of the notebook.
    RUN_DIR = Path("/kaggle/working/runs/t4")
    WORK    = Path("/kaggle/working")
else:
    RUN_DIR = Path.home() / "mhc1-runs" / "t4"
    WORK    = Path.home()

RUN_DIR.mkdir(parents=True, exist_ok=True)
REPO = WORK / "mhc1-boltz"
print(f"run dir: {RUN_DIR}")
print(f"repo:    {REPO}")

# A ~4 GB checkpoint lands here when val/lddt improves, plus last.ckpt.
# Free Drive is 15 GB total, so check before rather than after.
if ON_COLAB:
    free = shutil.disk_usage(RUN_DIR).free / 1024**3
    print(f"drive:   {free:.1f} GiB free  (need ~10 GiB: best + last + headroom)")

## 3. Code: repo, upstream Boltz, and the two-hunk patch

In [ ]:
# The repo does not vendor boltz-src/ -- it is a clean v1.0.0 tree plus our patch,
# so this reproduces it exactly. See reports/UPSTREAM_PATCHES.md.
def run(cmd, **kw):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, **kw)
    if r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r

if not REPO.exists():
    run(f"git clone --depth 1 https://github.com/SaifSyed08/mhc1-boltz.git {REPO}")
else:
    print(f"{REPO} exists, leaving it alone")

BOLTZ = REPO / "boltz-src"
if not BOLTZ.exists():
    run(f"git clone https://github.com/jwohlwend/boltz.git {BOLTZ}")
    run(f"git -C {BOLTZ} checkout v1.0.0")
    run(f"git -C {BOLTZ} apply ../patches/boltz-v1.0.0-training-path.patch")
    print("patch applied")
else:
    print("boltz-src exists, leaving it alone")

## 4. Dependencies

Colab and Kaggle both ship a CUDA torch, and it is the single biggest install in
the stack -- reinstalling it costs minutes and risks pulling a build that does not
match the driver. So: keep the host's torch, install `boltz` with `--no-deps`, and
add only what the *training* path actually imports.

Three of boltz's pins are deliberately skipped, same as in the laptop's
`.venv-gpu`:

* `numpy==1.26.3` -- stale. The pipeline has been running on numpy 2.x throughout.
* `dm-tree` -- never imported under `src/boltz/`.
* `biopython` -- only `data/parse/fasta.py`, which is not on the training path.

`fairscale` is **not** skippable: it is what `activation_checkpointing: true`
actually calls.

In [ ]:
DEPS = [
    "hydra-core==1.3.2",
    "pytorch-lightning==2.4.0",
    "fairscale==0.4.13",
    "omegaconf==2.3.1",
    "einops==0.8.0",
    "mashumaro==3.14",
    "modelcif==1.8",
    "rdkit",
    "scipy",
    "pandas",
]
run("pip install -q " + " ".join(f"'{d}'" for d in DEPS))
run(f"pip install -q --no-deps -e {BOLTZ}")

# `pip install -e` drops __editable__.boltz-1.0.0.pth into site-packages, and .pth
# files are read by site.py ONLY at interpreter startup. This kernel was already
# running, so it will not see boltz however well the install went. Boltz uses a
# src layout, so putting boltz-src/src on sys.path gets this process there
# directly -- no kernel restart, no reinstall.
import importlib, sys
BOLTZ_SRC = str(BOLTZ / "src")
if BOLTZ_SRC not in sys.path:
    sys.path.insert(0, BOLTZ_SRC)
importlib.invalidate_caches()

for mod in ["boltz", "pytorch_lightning", "hydra", "fairscale", "rdkit"]:
    importlib.import_module(mod)
print("imports ok in this kernel")

# The kernel is not what runs training, though -- train.py runs in a subprocess.
# That is the import that actually has to work, so check it where it happens.
probe = subprocess.run(
    [sys.executable, "-c", "import boltz, pytorch_lightning, fairscale; print(boltz.__file__)"],
    capture_output=True, text=True, cwd=str(BOLTZ),
)
if probe.returncode == 0:
    print(f"imports ok in a fresh subprocess: {probe.stdout.strip()}")
    PYTHONPATH_PREFIX = ""
else:
    # Editable install did not take. Hand the path to the subprocess explicitly
    # rather than trying to repair site-packages.
    print("subprocess import FAILED -- falling back to PYTHONPATH:")
    print(probe.stderr.strip()[-400:])
    PYTHONPATH_PREFIX = f"PYTHONPATH={BOLTZ_SRC} "

import torch
print("torch still:", torch.__version__, "| cuda:", torch.cuda.is_available())

### Optional: `trifast`

This is the one genuine upside of leaving Windows. `trifast` is a Triton kernel
for triangular attention, and Boltz soft-imports it
(`triangular_attention/primitives.py:46`). It could never be installed on the
laptop -- Triton has no Windows build -- and triangular attention is precisely
where the 8 GB runs died.

It is **off by default here** because 16 GB does not need it and an unvalidated
kernel swap is a bad thing to have running underneath a result you intend to
report. To try it: install it and add
`model.pairformer_args.use_trifast=true` to the command line in section 8 --
`pairformer_args` is splatted straight into `PairformerModule`
(`model/model.py:192`), so the flag reaches the layer. Then re-run the probe in
section 7 and compare peak VRAM and s/step against the numbers you already have.
Treat it as an experiment with a before/after, not as a default.

In [ ]:
INSTALL_TRIFAST = False  # flip to True to experiment; see section 7 for the A/B

if INSTALL_TRIFAST:
    run("pip install -q trifast")
    import importlib.util
    print("trifast importable:", importlib.util.find_spec("trifast") is not None)
else:
    print("skipped (default)")

## 5. Data: the bundle, then the assets

In [ ]:
import glob, tarfile, time

# Two shapes to handle, because Kaggle auto-extracts archives on upload: the
# dataset may hold the tarball, or it may hold the already-unpacked tree. Look
# for both rather than assuming.
def find_extracted():
    for hit in glob.glob("/kaggle/input/*/**/processed/structures", recursive=True):
        p = Path(hit)
        if any(p.glob("*.npz")):
            return p.parent.parent          # the dir containing processed/ and msa/
    return None

BUNDLE = EXTRACTED = None
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/mhc1-data-bundle.tar*", recursive=True)
    BUNDLE = hits[0] if hits else None
    if not BUNDLE:
        EXTRACTED = find_extracted()
elif ON_COLAB:
    BUNDLE = "/content/drive/MyDrive/mhc1/mhc1-data-bundle.tar.gz"  # <- edit if elsewhere

(REPO / "data").mkdir(parents=True, exist_ok=True)

if (REPO / "data" / "processed" / "structures").exists():
    print("data already staged")

elif EXTRACTED is not None:
    # Kaggle unpacked it for us. /kaggle/input is read-only, but the training
    # path only ever reads these, so link rather than burn 570 MB of working
    # quota on a copy.
    print(f"found an extracted tree at {EXTRACTED} -- linking")
    for sub in ("processed", "msa"):
        src, dst = EXTRACTED / sub, REPO / "data" / sub
        if dst.exists() or dst.is_symlink():
            continue
        try:
            dst.symlink_to(src, target_is_directory=True)
            print(f"  {dst} -> {src}")
        except OSError:
            shutil.copytree(src, dst)
            print(f"  copied {src} -> {dst} (symlink refused)")

elif BUNDLE and Path(BUNDLE).exists():
    print(f"unpacking {BUNDLE} -> {REPO}")
    t0 = time.time()
    with tarfile.open(BUNDLE) as tar:
        tar.extractall(REPO)
    print(f"done in {time.time()-t0:.0f}s")

else:
    if ON_KAGGLE:
        print("Nothing usable under /kaggle/input. What is actually there:")
        for d in sorted(glob.glob("/kaggle/input/*")):
            print(f"  {d}")
            for f in sorted(glob.glob(d + "/*"))[:10]:
                print(f"    {Path(f).name}")
    raise SystemExit(
        "Data bundle not found.\n"
        "  Build it on the laptop:  python src/make_cloud_bundle.py\n"
        "  Colab : upload to MyDrive/mhc1/ (or edit BUNDLE above)\n"
        "  Kaggle: create a Dataset from it, then Add Input on this notebook"
    )

n_struct = len(list((REPO / "data/processed/structures").glob("*.npz")))
n_msa    = len(list((REPO / "data/msa").glob("*.npz")))
print(f"structures: {n_struct}   msa: {n_msa}")
assert n_struct == 1084, f"expected 1084 structures, found {n_struct}"

In [ ]:
# The checkpoint (3.6 GB) and symmetry pickle (215 MB) come from their original
# hosts -- faster than pushing them through Drive, and src/fetch_assets.py already
# knows the URLs and verifies what it pulls.
assets = REPO / "data" / "assets"
if (assets / "boltz1_conf.ckpt").exists() and (assets / "symmetry.pkl").exists():
    print("assets already present")
else:
    t0 = time.time()
    run(f"cd {REPO} && python src/fetch_assets.py")
    print(f"fetched in {time.time()-t0:.0f}s")

for f in sorted(assets.iterdir()):
    print(f"  {f.name:<22} {f.stat().st_size/1024**3:.2f} GiB")

## 6. Sanity check: the pretrained baseline still reproduces

Optional, but worth doing once on a new machine. This is `validation_only` against
the 30-sample subset, and it should land near the numbers in `reports/BASELINE.md`
(~0.88 protein-protein lDDT, ~3.05 A RMSD). If it does not, the problem is the
environment, and you want to know that before spending hours training in it.

Validation is expensive here -- `sampling_steps: 200`, `diffusion_samples: 5`,
`symmetry_correction: true` -- so budget a couple of hours, or skip it and go
straight to section 7.

In [ ]:
RUN_BASELINE = False  # set True to reproduce the pretrained baseline first

if RUN_BASELINE:
    log = RUN_DIR / "baseline_t4.log"
    cmd = (
        f"cd {BOLTZ} && {PYTHONPATH_PREFIX}python scripts/train/train.py "
        f"../configs/mhc1_baseline_subset30.yaml "
        f"output={RUN_DIR / 'baseline'} 2>&1 | tee {log}"
    )
    run(cmd)
else:
    print("skipped")

## 7. Probe: does it fit, and how fast is it?

Do not start a multi-hour run on an assumption. This runs a handful of training
steps and reports two numbers:

* **peak VRAM** -- the arithmetic in `reports/GPU_REQUIREMENTS.md` predicts ~5.2 GB
  of persistent state for the frozen-trunk recipe (all weights 1.81 GB, gradients
  1.13 GB, Adam moments 2.25 GB), leaving ~10 GB for activations. The 8 GB card
  had ~2.9 GB for activations and died needing another 730 MB.
* **seconds per training batch** -- the number that decides whether the experiment
  is feasible at all, and it cannot be guessed from the 4060's timings. A T4 has
  no TF32, so fp32 matmuls run on plain CUDA cores at ~8.1 TFLOPS.

The next cell turns those into a budget for a 12 h session and a 30 h week.

In [ ]:
import re, threading

PROBE_STEPS = 8
probe_log = RUN_DIR / "probe.log"

# Training runs in a subprocess, so torch.cuda.max_memory_allocated() in THIS
# process would read 0. Poll nvidia-smi instead -- it also catches the CUDA
# context and any fragmentation, which is what actually has to fit.
peak_mib = [0]
stop = threading.Event()

def watch_vram():
    while not stop.is_set():
        try:
            out = subprocess.run(
                "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                shell=True, capture_output=True, text=True, timeout=5)
            peak_mib[0] = max(peak_mib[0], int(out.stdout.strip().split("\n")[0]))
        except Exception:
            pass
        stop.wait(2)

cmd = (
    f"cd {BOLTZ} && {PYTHONPATH_PREFIX}PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
    f"python scripts/train/train.py ../configs/mhc1_finetune_t4.yaml "
    f"output={RUN_DIR / 'probe'} "
    f"trainer.max_steps={PROBE_STEPS} "
    f"trainer.limit_val_batches=0 "
    f"disable_checkpoint=true"
)
print(f"$ {cmd}\n")

watcher = threading.Thread(target=watch_vram, daemon=True)
watcher.start()
t0 = time.time()
with open(probe_log, "w") as fh:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
        fh.write(line)
    p.wait()
elapsed = time.time() - t0
stop.set()
watcher.join(timeout=5)

print(f"\n--- probe finished in {elapsed:.0f}s (rc={p.returncode}) ---")
print(f"peak VRAM (nvidia-smi): {peak_mib[0]} MiB of {vram*1024:.0f} MiB "
      f"= {100*peak_mib[0]/(vram*1024):.0f}% of the card")

In [ ]:
# Read the probe back and turn it into a budget.
text = probe_log.read_text()

if "out of memory" in text.lower():
    print("OOM. It does not fit as configured. Levers, cheapest first:")
    print("  1. PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True   (already on)")
    print("  2. model.msa_args.offload_to_cpu=true, and the same for")
    print("     pairformer_args and score_model_args")
    print("  3. data.max_tokens=384                    <- CHANGES THE OBJECTIVE, note it")
    print("  4. model.training_args.diffusion_multiplicity=8   <- ditto")
else:
    # Lightning's progress bar carries the per-batch rate; fall back to wall clock.
    rates = re.findall(r"([\d.]+)\s*s/it", text)
    if rates:
        s_per_batch = float(rates[-1])
        src = "progress bar"
    else:
        s_per_batch = elapsed / PROBE_STEPS
        src = "wall clock, includes ~1-2 min startup -- pessimistic"

    ACCUM, SAMPLES = 16, 100     # accumulate_grad_batches, samples_per_epoch
    step_s  = s_per_batch * ACCUM
    epoch_h = s_per_batch * SAMPLES / 3600

    print(f"seconds per batch: {s_per_batch:.1f}   ({src})")
    print(f"per optimizer step (accum={ACCUM}): {step_s/60:.1f} min")
    print(f"per epoch ({SAMPLES} samples):      {epoch_h:.1f} h")
    print()
    print(f"one 12 h session:   ~{12/epoch_h:.1f} epochs, ~{12*3600/step_s:.0f} optimizer steps")
    print(f"a 30 h Kaggle week: ~{30/epoch_h:.1f} epochs, ~{30*3600/step_s:.0f} optimizer steps")
    print()
    print("Validation is NOT in these numbers and is expensive (sampling_steps=200,")
    print("diffusion_samples=5) -- it runs once per epoch over 30 samples.")

## 8. The run

`max_epochs: -1` plus a 12 h wall means this run will be interrupted. That is
fine, and it is designed for: the checkpoint callback writes `last.ckpt` into
`RUN_DIR`, and the cell below passes it back as `resume=` if it is there.

**So after a disconnect: re-run sections 1-5, then this cell.** It picks up where
it stopped. When `resume` is set, `pretrained` is ignored
(`scripts/train/train.py:130`), which is the correct behaviour -- you want the
optimizer state back, not a fresh load of the pretrained weights.

On Kaggle, use **Save & Run All (Commit)** rather than running interactively. The
job keeps going with the tab closed, which is the whole reason to prefer Kaggle.

In [ ]:
resume = RUN_DIR / "last.ckpt"
train_log = RUN_DIR / f"finetune_{time.strftime('%Y%m%d_%H%M%S')}.log"

parts = [
    f"cd {BOLTZ}",
    f"&& {PYTHONPATH_PREFIX}PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True",
    "python scripts/train/train.py ../configs/mhc1_finetune_t4.yaml",
    f"output={RUN_DIR}",
]
if resume.exists():
    parts.append(f"resume={resume}")
    print(f"RESUMING from {resume} ({resume.stat().st_size/1024**3:.1f} GiB)")
else:
    print("fresh start from the pretrained checkpoint")

cmd = " ".join(parts)
print(f"$ {cmd}\nlogging to {train_log}\n")

with open(train_log, "w") as fh:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
            fh.write(line)
            fh.flush()          # so the log survives a hard session kill
    except KeyboardInterrupt:
        p.terminate()
        print("\ninterrupted -- last.ckpt in RUN_DIR is the resume point")
    p.wait()
print(f"\nexit {p.returncode}")

## 9. What came out

`ValProgressDump` (our patch, `src/val_progress.py`) snapshots the running
validation metrics to `val_state.json` after every batch, so an interrupted
validation is not a total loss -- which matters a lot more on a 12 h session than
it did on the laptop.

In [ ]:
import json

vs = RUN_DIR / "val_state.json"
if vs.exists():
    print(json.dumps(json.loads(vs.read_text()), indent=2)[:2000])
else:
    print("no val_state.json yet -- validation has not run")

print("\ncheckpoints:")
for f in sorted(RUN_DIR.glob("*.ckpt")):
    print(f"  {f.name:<40} {f.stat().st_size/1024**3:.2f} GiB")

In [ ]:
# Copy the run artefacts back into the repo tree so they can be committed from the
# laptop. Checkpoints are deliberately NOT copied -- they are ~4 GB each.
dest = REPO / "reports" / "t4"
dest.mkdir(parents=True, exist_ok=True)
for pat in ("*.log", "val_state.json", "*.json"):
    for f in RUN_DIR.glob(pat):
        shutil.copy2(f, dest / f.name)
        print("copied", f.name)

print(f"\nDownload {dest} and commit it. On Kaggle everything under "
      f"/kaggle/working is already in the committed version's Output tab.")